# KernelForge — GPU session

**Settings → Accelerator → GPU T4** and **Internet → On**, then *Run All*.

Produces the GPU-verified dataset (`data/verified/dataset.jsonl`), the per-category pass/speedup report, and a baseline eval on held-out templates. Set `FINETUNE = True` to also train the QLoRA adapter and evaluate it on the same held-out ops (adds about an hour).

Everything lands in `/kaggle/working/kernelforge_outputs.zip` (Output panel → download).

In [ ]:
FINETUNE = False      # True: + QLoRA fine-tune and fine-tuned eval
EVAL_LIMIT = 25       # held-out examples per eval, round-robin across the 5 categories
BRANCH = "main"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!rm -rf KernelForge && git clone -q --depth 1 -b {BRANCH} https://github.com/Arsalan-05/KernelForge.git
%cd KernelForge
!pip install -q -r requirements-demo.txt
!python -c "import torch, triton; print('torch', torch.__version__, '| triton', triton.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
flags = f"--baseline-limit {EVAL_LIMIT}" + (" --finetune" if FINETUNE else "")
!python scripts/gpu_session.py {flags}

In [ ]:
import json
s = json.load(open("/kaggle/working/kernelforge_outputs/session.json"))
print(json.dumps(s["env"], indent=2))
for stage, r in s["stages"].items():
    print(f"{stage:<10} {r['status']}")
rep = s.get("dataset_report", {}).get("per_category", {})
for cat, r in rep.items():
    print(f"{cat:<18} {r}")